# 一键构建安卓APK\n点击上方菜单: **runtime → Run all** 即可自动构建\n构建完成后会自动下载APK到你的电脑

In [ ]:
# 1. 安装Buildozer和依赖\n!pip install buildozer cython virtualenv\n!sudo apt-get update -qq\n!sudo apt-get install -y -qq python3-pip autoconf libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo5 cmake libffi-dev libssl-dev openjdk-17-jdk zip unzip\nprint('依赖安装完成')

In [ ]:
# 2. 写入项目文件\nimport os, json\nos.makedirs('/content/app/java/com/detector', exist_ok=True)\n\n# main.py\nmain_py = '''\nimport os, time, numpy as np, cv2\nfrom collections import deque\nfrom kivy.app import App\nfrom kivy.uix.boxlayout import BoxLayout\nfrom kivy.uix.button import Button\nfrom kivy.uix.label import Label\nfrom kivy.uix.image import Image\nfrom kivy.uix.togglebutton import ToggleButton\nfrom kivy.clock import Clock\nfrom kivy.graphics.texture import Texture\nfrom kivy.core.window import Window\nimport subprocess\n\nclass EnemyDetector:\n    def __init__(self):\n        self._bg = cv2.createBackgroundSubtractorMOG2(200, 40, False)\n        self._trails = {}\n        self._next_id = 0\n    \n    def detect(self, frame):\n        h, w = frame.shape[:2]\n        display = frame.copy()\n        fg = self._bg.apply(frame)\n        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))\n        fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, kernel, iterations=2)\n        contours, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n        count = 0\n        cur_ids = set()\n        for c in contours:\n            area = cv2.contourArea(c)\n            if area < 200 or area > 60000: continue\n            x, y, bw, bh = cv2.boundingRect(c)\n            aspect = bh / max(bw, 1)\n            if aspect < 0.7 or aspect > 4.5: continue\n            if x < 10 or y < 10 or x + bw > w - 10 or y + bh > h - 10: continue\n            cx, cy = x + bw // 2, y + bh // 2\n            matched = None\n            for tid, trail in self._trails.items():\n                if trail:\n                    lx, ly, lw, lh = trail[-1]\n                    if abs(cx - (lx + lw // 2)) + abs(cy - (ly + lh // 2)) < 60:\n                        matched = tid; break\n            if matched is None: matched = self._next_id; self._next_id += 1\n            self._trails.setdefault(matched, deque(maxlen=20)).append((x, y, bw, bh))\n            cur_ids.add(matched)\n            cv2.rectangle(display, (x, y), (x + bw, y + bh), (0, 255, 0), 2)\n            cv2.putText(display, f'E', (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)\n            count += 1\n        stale = set(self._trails) - cur_ids\n        for s in stale: del self._trails[s]\n        return display, count\n    def reset(self):\n        self._bg = cv2.createBackgroundSubtractorMOG2(200, 40, False)\n        self._trails.clear()\n\nclass EnemyDetectorApp(App):\n    def build(self):\n        self._detector = EnemyDetector()\n        self._running = False\n        self._fps_times = deque(maxlen=30)\n        layout = BoxLayout(orientation='vertical', padding=8, spacing=4)\n        self._status = Label(text='就绪', size_hint=(1, 0.05), color=(0.5, 1, 0.5, 1))\n        layout.add_widget(self._status)\n        self._preview = Image(size_hint=(1, 0.85))\n        layout.add_widget(self._preview)\n        bl = BoxLayout(size_hint=(1, 0.07), spacing=4)\n        self._btn = ToggleButton(text='开始检测')\n        self._btn.bind(on_press=self._toggle)\n        bl.add_widget(self._btn)\n        bl.add_widget(Button(text='重置', on_press=lambda x: self._detector.reset()))\n        layout.add_widget(bl)\n        self._info = Label(text='目标: 0', size_hint=(1, 0.03), color=(0.7, 0.7, 0.7, 1))\n        layout.add_widget(self._info)\n        return layout\n    def _toggle(self, btn):\n        if btn.state == 'down':\n            self._running = True\n            Clock.schedule_interval(self._update, 1.0/15.0)\n        else:\n            self._running = False\n            Clock.unschedule(self._update)\n    def _update(self, dt):\n        if not self._running: return\n        t0 = time.time()\n        try: r = subprocess.run(['screencap', '-p'], capture_output=True, timeout=2)\n        except: return\n        if r.returncode != 0: return\n        frame = cv2.imdecode(np.frombuffer(r.stdout, np.uint8), cv2.IMREAD_COLOR)\n        if frame is None: return\n        display, count = self._detector.detect(frame)\n        h, w = display.shape[:2]\n        rgb = cv2.cvtColor(display, cv2.COLOR_BGR2RGB)\n        s = min(Window.width/w, Window.height*0.85/h, 1.0)\n        if s < 1.0: rgb = cv2.resize(rgb, (int(w*s), int(h*s)))\n        buf = cv2.flip(rgb, 0).tobytes()\n        tex = Texture.create(size=(rgb.shape[1], rgb.shape[0]), colorfmt='rgb')\n        tex.blit_buffer(buf, colorfmt='rgb', bufferfmt='ubyte')\n        self._preview.texture = tex\n        self._fps_times.append(time.time()-t0)\n        fps = len(self._fps_times)/sum(self._fps_times) if self._fps_times else 0\n        self._info.text = f'目标: {count} | FPS: {fps:.0f}'\n\nif __name__ == '__main__': EnemyDetectorApp().run()\n'''\nwith open('/content/app/main.py', 'w') as f: f.write(main_py)\n\n# buildozer.spec\nspec = '''[app]\ntitle = Enemy Detector\npackage.name = enemydetect\npackage.domain = com.detector\nsource.dir = .\nsource.include_exts = py,png,jpg\nversion = 1.0\nrequirements = python3,kivy,opencv,numpy\norientation = portrait\nfullscreen = 1\nandroid.permissions = INTERNET,SYSTEM_ALERT_WINDOW,FOREGROUND_SERVICE\nandroid.api = 33\nandroid.minapi = 26\nandroid.ndk = 25b\nandroid.sdk = 33\nandroid.arch = arm64-v8a\nandroid.allow_backup = True\np4a.branch = develop\np4a.bootstrap = sdl2\n\n[buildozer]\nlog_level = 2\nwarn_on_root = 1\n'''\nwith open('/content/app/buildozer.spec', 'w') as f: f.write(spec)\n\nprint('项目文件写入完成')\n!ls -la /content/app/

In [ ]:
# 3. 开始构建APK (需要10-20分钟, 首次需下载SDK/NDK)\nimport os\nos.chdir('/content/app')\n!buildozer android debug 2>&1 | tail -20\n\n# 构建完成后列出APK\n!find /content/app -name '*.apk' -exec ls -lh {} \\;

In [ ]:
# 4. 下载APK到电脑\nfrom google.colab import files\nimport glob\napks = glob.glob('/content/app/bin/*.apk')\nif apks:\n    for apk in apks:\n        files.download(apk)\n        print(f'下载: {apk}')\nelse:\n    print('APK未生成, 请检查上方构建日志')\n    !cat /content/app/.buildozer/android/platform/python-for-android/build.log 2>/dev/null | tail -30